# Lab Assignment 2 - Sequence Modelling NLP Part 2

Notebook ini sengaja dibuat simpel:

- `src/task1_text_generator.py` untuk Task 1.
- `src/task2_seq2seq.py` untuk Task 2 dan Task 3.
- `src/optional_task.py` untuk beam search dan pointer-generator.
- `src/common.py` untuk helper umum.
- `src/sample_data.py` untuk data contoh dan loader dataset publik.

## Setup

Cell pertama mengambil repo dari GitHub sebagai ZIP, menyalin folder `src`, lalu menghapus file sementara. Dependency seperti `torch`, `matplotlib`, dan `sacrebleu` diasumsikan sudah tersedia di environment/Colab/conda.

In [ ]:
# download src folder from GitHub if running this notebook in Google Colab
!wget -O repo.zip https://github.com/teranixbq/SequenceModellingNLP_Part2/archive/refs/heads/main.zip
!unzip -oq repo.zip
!rm -rf src
!cp -r SequenceModellingNLP_Part2-main/src src
!rm -rf repo.zip SequenceModellingNLP_Part2-main

In [ ]:
import matplotlib.pyplot as plt
import torch
from src.common import calculate_bleu, perplexity, set_seed
from src.optional_task import build_pointer_generator_model, translate_with_beam_search
from src.sample_data import SMALL_TRANSLATION_PAIRS, TEXT_CORPUS, load_csv_pairs, load_talpco_pairs_from_github
from src.task1_text_generator import (
    LSTMTextGenerator,
    SimpleRNNTextGenerator,
    build_char_vocab,
    generate_text,
    make_text_dataloader,
    train_text_generator,
)
from src.task2_seq2seq import (
    build_seq2seq_model,
    build_translation_data,
    make_translation_dataloader,
    train_seq2seq,
    translate_sentence,
)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## Dataset Translation

Dataset utama yang dipakai: TALPCo dari GitHub, bukan Hugging Face.

TALPCo punya file English dan Indonesian dengan urutan baris paralel:

- English: `eng/data_eng.txt`
- Indonesian: `ind/data_ind.txt`

Loader di `src/sample_data.py` membaca dua raw URL GitHub itu, lalu memasangkan baris ke-n English dengan baris ke-n Indonesian.

In [ ]:
# Dataset utama: GitHub TALPCo, bukan Hugging Face.
try:
    translation_pairs = load_talpco_pairs_from_github(max_samples=1000)
except Exception as error:
    print('Gagal download TALPCo, pakai data kecil dulu:', error)
    translation_pairs = SMALL_TRANSLATION_PAIRS

# Kalau pakai CSV Kaggle/Mendeley yang sudah di-upload ke Colab:
# translation_pairs = load_csv_pairs('/content/inmad.csv', 'english', 'indonesia', max_samples=1000)

len(translation_pairs), translation_pairs[:3]

## Task 1 - LSTM Text Generator

Requirement yang dipenuhi:

- memakai `nn.LSTM`,
- training dengan sequence input,
- generate text minimal 100 karakter,
- membandingkan Simple RNN vs LSTM.

In [ ]:
char_to_id, id_to_char = build_char_vocab(TEXT_CORPUS)
text_loader = make_text_dataloader(TEXT_CORPUS, char_to_id, sequence_length=60, batch_size=32)

simple_rnn = SimpleRNNTextGenerator(len(id_to_char), embedding_size=48, hidden_size=96).to(device)
lstm_model = LSTMTextGenerator(len(id_to_char), embedding_size=48, hidden_size=96).to(device)

rnn_losses = train_text_generator(simple_rnn, text_loader, epochs=15, learning_rate=0.003, device=device)
lstm_losses = train_text_generator(lstm_model, text_loader, epochs=15, learning_rate=0.003, device=device)

print(f'Simple RNN loss: {rnn_losses[-1]:.4f} | perplexity: {perplexity(rnn_losses[-1]):.2f}')
print(f'LSTM loss      : {lstm_losses[-1]:.4f} | perplexity: {perplexity(lstm_losses[-1]):.2f}')

In [ ]:
plt.plot(rnn_losses, label='Simple RNN')
plt.plot(lstm_losses, label='LSTM')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

print(generate_text(lstm_model, 'natural language', char_to_id, id_to_char, generated_length=180, device=device))

## Task 2 - Encoder Decoder LSTM

Requirement yang dipenuhi:

- encoder bidirectional LSTM,
- decoder LSTM dengan teacher forcing,
- embedding layer + LSTM,
- `pack_padded_sequence` untuk variable sequence length.

In [ ]:
data = build_translation_data(translation_pairs, train_ratio=0.85, seed=42)
source_token_to_id = data['source_token_to_id']
target_token_to_id = data['target_token_to_id']
target_id_to_token = data['target_id_to_token']

train_loader = make_translation_dataloader(
    data['train_pairs'],
    source_token_to_id,
    target_token_to_id,
    batch_size=16,
    shuffle=True,
)

basic_model = build_seq2seq_model(
    source_token_to_id,
    target_token_to_id,
    embedding_size=64,
    hidden_size=96,
    decoder_type='basic',
).to(device)

basic_losses = train_seq2seq(
    basic_model,
    train_loader,
    epochs=40,
    learning_rate=0.002,
    target_pad_id=target_token_to_id['<pad>'],
    device=device,
    teacher_forcing_ratio=0.6,
)

print(f'Encoder-decoder final loss: {basic_losses[-1]:.4f}')

## Task 3 - Bahdanau Attention

Model berikut memakai additive attention dan dapat divisualisasikan.

In [ ]:
attention_model = build_seq2seq_model(
    source_token_to_id,
    target_token_to_id,
    embedding_size=64,
    hidden_size=96,
    decoder_type='attention',
).to(device)

attention_losses = train_seq2seq(
    attention_model,
    train_loader,
    epochs=40,
    learning_rate=0.002,
    target_pad_id=target_token_to_id['<pad>'],
    device=device,
    teacher_forcing_ratio=0.6,
)

print(f'Attention model final loss: {attention_losses[-1]:.4f}')

In [ ]:
test_sources = [source for source, _ in data['test_pairs']]
test_references = [target for _, target in data['test_pairs']]

basic_predictions = [
    translate_sentence(basic_model, source, source_token_to_id, target_id_to_token, device)['translation']
    for source in test_sources
]
attention_predictions = [
    translate_sentence(attention_model, source, source_token_to_id, target_id_to_token, device)['translation']
    for source in test_sources
]

print(f'BLEU without attention: {calculate_bleu(test_references, basic_predictions):.2f}')
print(f'BLEU with attention   : {calculate_bleu(test_references, attention_predictions):.2f}')

In [ ]:
def show_attention(sentence):
    result = translate_sentence(attention_model, sentence, source_token_to_id, target_id_to_token, device)
    attention = result['attention']
    if attention is None:
        print('Model ini tidak punya attention weights.')
        return

    source_tokens = result['source_tokens']
    predicted_tokens = result['predicted_tokens']
    matrix = attention[: len(predicted_tokens), : len(source_tokens)].numpy()

    print('EN  :', sentence)
    print('PRED:', result['translation'])
    plt.figure(figsize=(max(5, len(source_tokens)), max(3, len(predicted_tokens) * 0.6)))
    plt.imshow(matrix, cmap='viridis', aspect='auto')
    plt.xticks(range(len(source_tokens)), source_tokens, rotation=45, ha='right')
    plt.yticks(range(len(predicted_tokens)), predicted_tokens)
    plt.xlabel('Source words')
    plt.ylabel('Predicted words')
    plt.colorbar()
    plt.tight_layout()
    plt.show()

for source, _ in data['train_pairs'][:3]:
    show_attention(source)

## Bonus 1 - Beam Search

Beam search dipakai saat decoding, bukan saat training.

In [ ]:
example_sentence = test_sources[0] if test_sources else data['train_pairs'][0][0]

greedy_result = translate_sentence(attention_model, example_sentence, source_token_to_id, target_id_to_token, device)
beam_result = translate_with_beam_search(
    attention_model,
    example_sentence,
    source_token_to_id,
    target_id_to_token,
    device,
    beam_size=3,
)

print('Source:', example_sentence)
print('Greedy:', greedy_result['translation'])
print('Beam  :', beam_result['translation'])

## Bonus 2 - Pointer Generator Network

Pointer-generator menggabungkan vocabulary distribution dan copy distribution dari attention weights. Ini berguna saat ada nama orang, lokasi, atau token source yang sebaiknya disalin.

In [ ]:
pointer_model = build_pointer_generator_model(
    source_token_to_id,
    target_token_to_id,
    embedding_size=64,
    hidden_size=96,
    copy_map=data['copy_map'],
).to(device)

pointer_losses = train_seq2seq(
    pointer_model,
    train_loader,
    epochs=40,
    learning_rate=0.002,
    target_pad_id=target_token_to_id['<pad>'],
    device=device,
    teacher_forcing_ratio=0.6,
)

pointer_result = translate_with_beam_search(
    pointer_model,
    example_sentence,
    source_token_to_id,
    target_id_to_token,
    device,
    beam_size=3,
)
print(f'Pointer-generator final loss: {pointer_losses[-1]:.4f}')
print('Pointer + beam:', pointer_result['translation'])